# 87 — Biological Fingerprinting: ChEMBL Nuclear Receptor Panel

**Key insight:** A molecule's *activity profile across related nuclear receptors* encodes biological context.

PPARg, FXR, RXRa, LXRa, VDR all share structural features with PXR. Compounds that activate these NRs are enriched for PXR agonists.

Strategy:
1. Train one LGBM binary classifier per NR (active = pEC50 ≥ 5.0) on ChEMBL+BindingDB data
2. Predict P(active) for all PXR train + test compounds → biological fingerprint (5-6 floats)
3. Concatenate bio-FP with combined Morgan+RDKit → train PXR LGBM

This is orthogonal to structural fingerprints — captures mechanistic signal.

In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch, standardize_smiles, compute_physchem
from pxr.paths import DATA_PROCESSED, DATA_EXTERNAL, SUBMISSIONS
SEED = 42; N_FOLDS = 5
LGBM = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
            min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
            reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)


In [2]:
def full_metrics(y_true, y_pred, cp=None, label=""):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    msk = np.isfinite(yt) & np.isfinite(yp); yt, yp = yt[msk], yp[msk]
    mae = float(np.mean(np.abs(yt-yp)))
    rae_v = mae / float(np.mean(np.abs(yt-yt.mean()))) if yt.std()>0 else float("nan")
    r2  = 1-np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2) if yt.std()>0 else float("nan")
    pr, _ = stats.pearsonr(yt, yp); sp, _ = stats.spearmanr(yt, yp)
    kt, _ = stats.kendalltau(yt, yp)
    m = dict(RAE=rae_v, MAE=mae, R2=float(r2), Pearson=float(pr),
             Spearman=float(sp), Kendall=float(kt))
    if cp is not None and hasattr(cp, "iterrows") and len(cp) > 0:
        c=t=0
        for _,row in cp.iterrows():
            ia,ii = int(row.get("idx_active",-1)), int(row.get("idx_inactive",-1))
            if 0<=ia<len(yp) and 0<=ii<len(yp): c+=int(yp[ia]>yp[ii]); t+=1
        m["Cliff_acc"] = c/t if t else float("nan")
    if label:
        ca = f"  Cliff={m.get('Cliff_acc',float('nan')):.3f}" if "Cliff_acc" in m else ""
        print(f"  [{label}] RAE={rae_v:.4f} MAE={mae:.4f} R²={r2:.4f} "
              f"r={pr:.4f} ρ={sp:.4f} τ={kt:.4f}{ca}")
    return m


In [3]:
tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, N_FOLDS, SEED)
active_mask = y_tr >= 5.5
X_tr = impute(combined(tr["smiles"].tolist()))
X_te = impute(combined(te["smiles"].tolist()))
fps_tr = morgan_fp_batch(tr["smiles"].tolist()).astype(np.float32)
fps_te = morgan_fp_batch(te["smiles"].tolist()).astype(np.float32)
cliff_pairs = (pd.read_parquet(DATA_PROCESSED/"cliff_pairs.parquet")
               if (DATA_PROCESSED/"cliff_pairs.parquet").exists() else pd.DataFrame())
# Build idx_active / idx_inactive
if len(cliff_pairs) > 0:
    s2i = {s:i for i,s in enumerate(tr["smiles"].tolist())}
    ac = "cliff_active_smiles" if "cliff_active_smiles" in cliff_pairs.columns else "smiles_a"
    ic = "cliff_inactive_smiles" if "cliff_inactive_smiles" in cliff_pairs.columns else "smiles_b"
    cliff_pairs["idx_active"]   = cliff_pairs[ac].map(s2i)
    cliff_pairs["idx_inactive"] = cliff_pairs[ic].map(s2i)
    cliff_pairs = cliff_pairs.dropna(subset=["idx_active","idx_inactive"])
    cliff_pairs[["idx_active","idx_inactive"]] = cliff_pairs[["idx_active","idx_inactive"]].astype(int)
print(f"Train {len(tr):,}  Test {len(te):,}  Cliffs {len(cliff_pairs)}")


Train 4,139  Test 513  Cliffs 0


In [4]:
from pxr.chem import to_inchikey

# Load ChEMBL + BindingDB NR data
chembl = pd.read_parquet(DATA_EXTERNAL/"chembl_nr_extended.parquet")
bdb    = pd.read_parquet(DATA_EXTERNAL/"bindingdb_nr_data.parquet")
nr_all = pd.concat([chembl, bdb], ignore_index=True)
nr_all["ik"] = nr_all["smiles"].map(to_inchikey)
nr_all = nr_all.dropna(subset=["smiles","pec50","target_name"])
nr_all["active"] = (nr_all["pec50"] >= 5.0).astype(int)

NR_TARGETS = ["PPARg","FXR","RXRa","LXRa","VDR"]
print("NR data counts:")
for t in NR_TARGETS:
    sub = nr_all[nr_all.target_name == t]
    print(f"  {t}: {len(sub):,} (active={sub.active.sum():,})")


NR data counts:
  PPARg: 6,795 (active=6,312)
  FXR: 3,641 (active=3,345)
  RXRa: 2,425 (active=2,320)
  LXRa: 2,029 (active=1,868)
  VDR: 1,001 (active=791)


In [5]:
# Train one binary LGBM classifier per NR → biological fingerprint
LGBM_CLS = dict(n_estimators=300, num_leaves=31, learning_rate=0.1,
                min_child_samples=5, subsample=0.8, colsample_bytree=0.8,
                random_state=SEED, verbose=-1, n_jobs=4)

tr_iks = tr["smiles"].map(to_inchikey).values
te_iks = te["smiles"].map(to_inchikey).values
fps_tr32 = fps_tr.astype(np.float32)
fps_te32 = fps_te.astype(np.float32)

bio_fp_tr = np.zeros((len(tr), len(NR_TARGETS)), dtype=np.float32)
bio_fp_te = np.zeros((len(te), len(NR_TARGETS)), dtype=np.float32)

for j, target in enumerate(NR_TARGETS):
    sub = nr_all[nr_all.target_name == target].dropna(subset=["smiles"])
    sub = sub.drop_duplicates("ik")
    X_nr = impute(combined(sub["smiles"].tolist()))
    y_nr = sub["active"].values.astype(int)
    if y_nr.sum() < 20 or (1-y_nr).sum() < 20:
        print(f"  {target}: too few labels — skipping"); continue

    m_cls = lgb.LGBMClassifier(**LGBM_CLS)
    m_cls.fit(X_nr, y_nr)
    bio_fp_tr[:, j] = m_cls.predict_proba(X_tr)[:, 1]
    bio_fp_te[:, j] = m_cls.predict_proba(X_te)[:, 1]
    print(f"  {target}: trained on {len(sub):,} cmpds  "
          f"→ PXR-train mean P(active)={bio_fp_tr[:,j].mean():.3f}", flush=True)

print(f"\nBio-fingerprint shape: {bio_fp_tr.shape}")
print(pd.DataFrame(bio_fp_tr, columns=NR_TARGETS).describe().round(3).to_string())


  PPARg: trained on 4,307 cmpds  → PXR-train mean P(active)=0.756


  FXR: trained on 3,185 cmpds  → PXR-train mean P(active)=0.949


  RXRa: trained on 1,364 cmpds  → PXR-train mean P(active)=0.786


  LXRa: trained on 1,174 cmpds  → PXR-train mean P(active)=0.916


  VDR: trained on 523 cmpds  → PXR-train mean P(active)=0.472



Bio-fingerprint shape: (4139, 5)
          PPARg       FXR      RXRa      LXRa       VDR
count  4139.000  4139.000  4139.000  4139.000  4139.000
mean      0.756     0.949     0.786     0.916     0.472
std       0.340     0.154     0.341     0.206     0.414
min       0.000     0.000     0.000     0.001     0.000
25%       0.623     0.985     0.723     0.964     0.027
50%       0.950     0.998     0.988     0.997     0.413
75%       0.994     1.000     0.999     1.000     0.950
max       1.000     1.000     1.000     1.000     1.000


In [6]:
# Augment features: combined + bio-FP
X_bio_tr = np.hstack([X_tr, bio_fp_tr])
X_bio_te = np.hstack([X_te, bio_fp_te])

# Also: just-bio-FP model (interpretability)
oof_bio_only = np.full(len(y_tr), np.nan)
oof_aug = np.full(len(y_tr), np.nan)

for fold, (tr_idx, va_idx) in enumerate(splits):
    # bio-FP only
    m1 = lgb.train(LGBM, lgb.Dataset(bio_fp_tr[tr_idx], label=y_tr[tr_idx]),
                   valid_sets=[lgb.Dataset(bio_fp_tr[va_idx], label=y_tr[va_idx])],
                   callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(-1)])
    oof_bio_only[va_idx] = m1.predict(bio_fp_tr[va_idx])

    # combined + bio-FP
    m2 = lgb.train(LGBM, lgb.Dataset(X_bio_tr[tr_idx], label=y_tr[tr_idx]),
                   valid_sets=[lgb.Dataset(X_bio_tr[va_idx], label=y_tr[va_idx])],
                   callbacks=[lgb.early_stopping(50,verbose=False), lgb.log_evaluation(-1)])
    oof_aug[va_idx] = m2.predict(X_bio_tr[va_idx])
    print(f"  fold {fold+1}  aug_RAE={rae(y_tr[va_idx], oof_aug[va_idx]):.4f}", flush=True)

m_bio = full_metrics(y_tr, oof_bio_only, cliff_pairs, "bio_fp_only")
m_aug = full_metrics(y_tr, oof_aug, cliff_pairs, "combined+bio_fp")
print("\n" + pd.DataFrame([m_bio, m_aug], index=["bio_only","augmented"]).round(4).to_string())


  fold 1  aug_RAE=0.4920


  fold 2  aug_RAE=0.5772


  fold 3  aug_RAE=0.5950


  fold 4  aug_RAE=0.5697


  fold 5  aug_RAE=0.6014


  [bio_fp_only] RAE=0.7944 MAE=0.7228 R²=0.2932 r=0.5421 ρ=0.4451 τ=0.3084
  [combined+bio_fp] RAE=0.5621 MAE=0.5114 R²=0.6021 r=0.7759 ρ=0.7311 τ=0.5384

              RAE     MAE      R2  Pearson  Spearman  Kendall
bio_only   0.7944  0.7228  0.2932   0.5421    0.4451   0.3084
augmented  0.5621  0.5114  0.6021   0.7759    0.7311   0.5384


In [7]:
m_final = lgb.train(LGBM, lgb.Dataset(X_bio_tr, label=y_tr), callbacks=[lgb.log_evaluation(-1)])
te_preds = np.clip(m_final.predict(X_bio_te), y_tr.min()-0.5, y_tr.max()+0.5)
np.save(DATA_PROCESSED/"oof_bio_nr_fingerprint.npy", oof_aug)
np.save(DATA_PROCESSED/"te_oof_bio_nr_fingerprint.npy", te_preds)
np.save(DATA_PROCESSED/"bio_fp_tr.npy", bio_fp_tr)
np.save(DATA_PROCESSED/"bio_fp_te.npy", bio_fp_te)
sub = pd.DataFrame({"Molecule Name": te["name"].values, "pEC50": te_preds})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"87_bio_nr_fingerprint.csv"; sub.to_csv(p, index=False)
print(f"Saved {p}")
print(f"Test: min={te_preds.min():.2f} med={np.median(te_preds):.2f} max={te_preds.max():.2f}")


Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\87_bio_nr_fingerprint.csv
Test: min=2.01 med=4.95 max=5.93
